In [0]:
df = spark.read.parquet("dbfs:/mnt/light/data/world_bank_global_postings/version=5/2025/07/09/18_00_44/*.parquet")

In [0]:
from pyspark.sql.functions import col, year, datediff, avg, count, sum

# Filter for U.S. and date
df = df.filter((col('nation_name') == "United States") & (col('posted') >= "2018-01-01"))

# Calculate job ad duration
df = df.withColumn("closing_time", datediff(col("expired"), col("posted")))

# Extract year
df = df.withColumn("year", year(col("posted")))

# Flag for valid NAICS codes
df = df.withColumn("has_valid_naics", (col("naics2") != 99).cast("int"))

# Perform all aggregations in a single .agg()
new_df = (
    df.groupBy("year", "laa_admin_area_2")
      .agg(
          avg("closing_time").alias("avg_closing_time"),
          count("*").alias("total_jobs"),
          sum("has_valid_naics").alias("jobs_with_valid_naics")
      )
      .withColumn("share_with_naics", col("jobs_with_valid_naics") / col("total_jobs"))
      .withColumnRenamed("laa_admin_area_2", "county_id")
)

display(new_df)


year,county_id,avg_closing_time,total_jobs,jobs_with_valid_naics,share_with_naics
2021,US48073,61.33125,2560,2027,0.791796875
2021,US35013,67.12786479450301,23431,18380,0.7844308821646536
2019,US18081,60.62382267989451,15926,13200,0.8288333542634685
2019,US31155,56.51848049281314,974,598,0.6139630390143738
2019,US13039,53.089563019140485,2769,2281,0.8237630913687252
2020,US42027,63.75258726539204,11402,10497,0.9206279600070163
2020,US21067,69.31534900001591,62851,51608,0.8211166091231643
2020,US56003,47.15560640732266,437,401,0.9176201372997712
2022,US47061,61.93624161073826,596,489,0.8204697986577181
2022,US46093,59.090425531914896,940,731,0.7776595744680851
